# Patrón de Comportamiento: Visitor

## Introducción
El patrón Visitor permite definir nuevas operaciones sobre una estructura de objetos sin cambiar las clases de los objetos sobre los que opera.

## Objetivos
- Comprender cómo separar algoritmos de las estructuras de datos.
- Identificar cuándo es útil el patrón Visitor.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: Sistema de impuestos para diferentes productos**
Un sistema de facturación puede aplicar diferentes impuestos a productos sin modificar las clases de producto.

**¿Dónde se usa en proyectos reales?**
En compiladores, sistemas de facturación, procesamiento de árboles, etc.

## Sin patrón Visitor (forma errónea)
Las operaciones se implementan dentro de las clases de los objetos, dificultando la extensión.

In [1]:
class Producto:
    def __init__(self, nombre: str, precio: float) -> None:
        self.nombre = nombre
        self.precio = precio
    def calcular_impuesto(self) -> float:
        return self.precio * 0.16

## Con patrón Visitor (forma correcta)
Las operaciones se implementan en visitantes separados.

In [2]:
class Visitor:
    def visitar(self, producto: 'Producto') -> float:
        pass

class ImpuestoNormal(Visitor):
    def visitar(self, producto: 'Producto') -> float:
        return producto.precio * 0.16

class ImpuestoReducido(Visitor):
    def visitar(self, producto: 'Producto') -> float:
        return producto.precio * 0.08

class Producto:
    def __init__(self, nombre: str, precio: float) -> None:
        self.nombre = nombre
        self.precio = precio
    def aceptar(self, visitor: Visitor) -> float:
        return visitor.visitar(self)

prod = Producto('Libro', 100)
print(prod.aceptar(ImpuestoNormal()))
print(prod.aceptar(ImpuestoReducido()))

16.0
8.0


## UML del patrón Visitor
```plantuml
@startuml
class Producto {
    + aceptar(visitor)
}
interface Visitor {
    + visitar(producto)
}
Visitor <|.. ImpuestoNormal
Visitor <|.. ImpuestoReducido
Producto --> Visitor
@enduml
```

## Otro ejemplo de la vida real: Exportar un documento estructurado a varios formatos
**Contexto:** un editor de documentos (estilo Sphinx o docutils) representa el contenido como nodos (`Titulo`, `Parrafo`, etc.) y necesita poder exportarlo a HTML, Markdown, PDF... Agregar un formato nuevo no debería obligar a modificar cada clase de nodo del documento.

### Sin patrón (forma errónea)
Cada nodo implementa un método por formato de exportación (`a_html`, `a_markdown`...). Agregar un formato nuevo obliga a tocar TODAS las clases de nodo existentes.

In [3]:
class Titulo:
    def __init__(self, texto: str) -> None:
        self.texto = texto
    def a_html(self) -> str:
        return f'<h1>{self.texto}</h1>'
    def a_markdown(self) -> str:
        return f'# {self.texto}'

class Parrafo:
    def __init__(self, texto: str) -> None:
        self.texto = texto
    def a_html(self) -> str:
        return f'<p>{self.texto}</p>'
    def a_markdown(self) -> str:
        return self.texto

# Agregar un formato "a_pdf" obligaría a tocar Titulo, Parrafo y cualquier otro nodo existente
titulo = Titulo('Guía de Python')
print(titulo.a_html())
print(titulo.a_markdown())

<h1>Guía de Python</h1>
# Guía de Python


### Con patrón (forma correcta)
Cada nodo solo sabe `aceptar(visitor)`. Toda la lógica de exportación vive en visitantes separados (`ExportadorHTML`, `ExportadorMarkdown`); un formato nuevo es un visitante nuevo, sin tocar los nodos.

In [4]:
class NodoDocumento:
    def aceptar(self, visitor: object) -> str:
        raise NotImplementedError

class Titulo(NodoDocumento):
    def __init__(self, texto: str) -> None:
        self.texto = texto
    def aceptar(self, visitor: object) -> str:
        return visitor.visitar_titulo(self)

class Parrafo(NodoDocumento):
    def __init__(self, texto: str) -> None:
        self.texto = texto
    def aceptar(self, visitor: object) -> str:
        return visitor.visitar_parrafo(self)


class ExportadorHTML:
    def visitar_titulo(self, nodo: Titulo) -> str:
        return f'<h1>{nodo.texto}</h1>'
    def visitar_parrafo(self, nodo: Parrafo) -> str:
        return f'<p>{nodo.texto}</p>'

class ExportadorMarkdown:
    def visitar_titulo(self, nodo: Titulo) -> str:
        return f'# {nodo.texto}'
    def visitar_parrafo(self, nodo: Parrafo) -> str:
        return nodo.texto


documento = [Titulo('Guía de Python'), Parrafo('Este capítulo cubre los patrones de diseño.')]

exportador_html = ExportadorHTML()
for nodo in documento:
    print(nodo.aceptar(exportador_html))

exportador_md = ExportadorMarkdown()
for nodo in documento:
    print(nodo.aceptar(exportador_md))

<h1>Guía de Python</h1>
<p>Este capítulo cubre los patrones de diseño.</p>
# Guía de Python
Este capítulo cubre los patrones de diseño.


### UML del ejemplo de exportación de documentos
```plantuml
@startuml
abstract class NodoDocumento {
    + aceptar(visitor)
}
class Titulo
class Parrafo
NodoDocumento <|-- Titulo
NodoDocumento <|-- Parrafo

interface Visitor {
    + visitar_titulo(nodo)
    + visitar_parrafo(nodo)
}
class ExportadorHTML
class ExportadorMarkdown
Visitor <|.. ExportadorHTML
Visitor <|.. ExportadorMarkdown
NodoDocumento --> Visitor
@enduml
```

### ¿Dónde más se usa Visitor?
- **Exportadores de documentos/AST:** exactamente este ejemplo — Sphinx, docutils o los transpiladores de Markdown a HTML recorren un árbol de nodos con visitantes.
- **Compiladores:** el árbol de sintaxis abstracta (AST) de un lenguaje se recorre con distintos visitantes para type-checking, optimización o generación de código.
- **Sistemas de facturación con impuestos:** el ejemplo con el que abre este notebook — aplicar distintas reglas de impuestos a productos sin modificar la clase `Producto`.
- **Linters y analizadores de código estático:** herramientas como ESLint recorren el AST del código fuente con "reglas" que son, en esencia, visitantes.
- **Serialización a múltiples formatos:** convertir una estructura de datos (árbol de un formulario, una escena 3D) a JSON, XML o un formato binario, cada uno como un visitante distinto.

**Ejercicio de reflexión:** si agregas un nuevo tipo de nodo `ListaItems` al documento, ¿qué tendrías que modificar: las clases `NodoDocumento`/`Titulo`/`Parrafo`, o los visitantes `ExportadorHTML`/`ExportadorMarkdown`? Relaciona tu respuesta con la principal desventaja de Visitor frente a Composite.

## Actividad
Crea un visitante que calcule descuentos para diferentes productos sin modificar sus clases.

---
## Explicación de conceptos clave
- **Separación de algoritmos:** Permite agregar operaciones sin modificar las clases de los objetos.
- **Extensibilidad:** Se pueden agregar nuevos visitantes fácilmente.
- **Aplicación en la vida real:** Útil en compiladores, facturación y procesamiento de árboles.

## Conclusión
El patrón Visitor es ideal para agregar operaciones a estructuras de objetos complejas sin modificar su código.